# 04 — Embeddings and hybrid retrieval

Load the enriched canonical menu persisted by notebook 02, index one document per item or variant, and inspect grounded retrieval candidates. This notebook runs independently of earlier notebook variables.

Set `DATABASE_URL`, `AI_PROVIDER`, and the selected provider’s API key in `.env`. Install `requirements-storage.txt` and `requirements-enrichment.txt`. Indexing and search cells call the selected AI provider; indexing re-embeds the menu on every run. See [STORAGE.md](STORAGE.md) for setup.

Filters below are explicit inputs. Natural-language intent extraction, allergen handling, fallback, and final response generation come later.

In [6]:
import os
from dataclasses import asdict
from decimal import Decimal
from dotenv import load_dotenv

from ai_adapters import create_ai_adapter
from menu_embeddings import AIEmbeddingProvider
from menu_retrieval import RetrievalFilters, retrieve_candidates
from menu_storage import PostgresMenuStore, build_embedding_documents, index_menu
from menu_validation import validate_menu

load_dotenv(override=True)
MENU_ID = os.environ.get("MENU_ID", "1c037ca1-b41a-4482-b9c7-85ecee7ab3e1")
store = PostgresMenuStore.connect(os.environ["DATABASE_URL"])
store.initialize()
menu = store.load_menu(MENU_ID)
if menu is None:
    raise RuntimeError("Menu not found. Persist it with notebook 02, or change MENU_ID.")
validate_menu(menu).raise_for_errors()
print(f"Loaded {len(menu.items)} items for {menu.restaurant_name} ({menu.currency})")

Loaded 204 items for Example restaurant (GBP)


## Inspect and index

Descriptions and semantic tags may be inferred. Dietary filters use only canonical verified attributes. The price shown in embedding text is the base/variant price; retrieval calculates required-choice costs separately.

In [7]:
documents = build_embedding_documents(menu)
print(f"{len(documents)} purchasable item/variant documents")
for item_id, variant_id, text, metadata in documents[:3]:
    print(item_id, variant_id)
    print(text)
    print(metadata)
    print()

219 purchasable item/variant documents
ef02fbf0-0691-48d1-a080-9d2f7c0fcf1d None
Blanco Candidato 250ml
Whites
Inferred description: A crisp white wine, often light-bodied with refreshing citrus and floral notes, offering a dry and clean finish.
Semantic attributes: refreshing
Price: GBP 6.75
{'menu_id': '1c037ca1-b41a-4482-b9c7-85ecee7ab3e1', 'item_id': 'ef02fbf0-0691-48d1-a080-9d2f7c0fcf1d', 'variant_id': None, 'currency': 'GBP', 'price': '6.75', 'availability': 'unavailable', 'category_ids': ['69bd9477-d325-4687-9d03-f9e05f49169f'], 'categories': ['Whites'], 'dietary_attributes': []}

4638ae68-a21d-4a8e-8cf1-3af80b141d35 None
Pinot Grigio BTL
Whites
Inferred description: A popular dry white wine, typically light-bodied with bright acidity, often featuring notes of green apple, pear, and a hint of minerality.
Semantic attributes: refreshing
Price: GBP 22.0
{'menu_id': '1c037ca1-b41a-4482-b9c7-85ecee7ab3e1', 'item_id': '4638ae68-a21d-4a8e-8cf1-3af80b141d35', 'variant_id': None, 'curre

In [8]:
provider = AIEmbeddingProvider(create_ai_adapter())
indexed_count = index_menu(menu, provider, store)
print(f"Indexed {indexed_count} documents with {provider.model_id} ({provider.dimensions} dimensions)")

Indexed 219 documents with openai:text-embedding-3-small:768:retrieval-v1 (768 dimensions)


## Compare with an open model (Qwen3)

`Qwen/Qwen3-Embedding-0.6B` runs locally: no API key, no per-query network call. The first run downloads about 1.2 GB of weights to the Hugging Face cache. Its vectors are truncated to 768 dimensions, the same size as the paid index, and stored under their own `qwen:` `model_id`, so both indexes live side by side.

The comparison uses vector search only (no lexical fusion), so differences come from the embedding models. Fill `EXPECTED` with item IDs you consider correct to get a recall@5 score.

With `AI_PROVIDER=qwen` in `.env`, the indexing cell above already uses Qwen3 and this comparison shows the same model twice.

In [ ]:
%pip install -q "sentence-transformers>=3"

In [4]:
import time
# Same adapter contract as OpenAI/Gemini; QWEN_EMBEDDING_MODEL overrides the model.
local_provider = AIEmbeddingProvider(create_ai_adapter("qwen", batch_size=16))
started = time.perf_counter()
local_count = index_menu(menu, local_provider, store)
print(f"Indexed {local_count} documents with {local_provider.model_id} in {time.perf_counter() - started:.1f}s")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6321.27it/s]
/Users/hassankhan/projects/business-projects/order-assistant/ai_adapters/qwen.py:25: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  native = client.get_sentence_embedding_dimension()


Indexed 219 documents with qwen:Qwen/Qwen3-Embedding-0.6B:768:retrieval-v1 in 16.7s


In [9]:
COMPARE_QUERIES = [
    "something sweet and light",
    "spicy chicken",
    "a cold drink",
    "vegetarian main course",
    "something to share with friends",
]
# Optional ground truth: query -> item IDs that a good answer should include.
EXPECTED: dict[str, set[str]] = {}
names = {item.id: item.name for item in menu.items}

def semantic_top(provider, query, k=5):
    started = time.perf_counter()
    vector = provider.embed_query(query)
    elapsed_ms = (time.perf_counter() - started) * 1000
    hits = store.search(menu_id=menu.id, model_id=provider.model_id, query_embedding=vector, limit=k)
    return [hit.item_id for hit in hits], elapsed_ms

recall = {provider.model_id: [], local_provider.model_id: []}
for query in COMPARE_QUERIES:
    print(f"\n{query!r}")
    for candidate_provider in (provider, local_provider):
        item_ids, elapsed_ms = semantic_top(candidate_provider, query)
        print(f"  {candidate_provider.model_id} ({elapsed_ms:.0f} ms): {[names[i] for i in item_ids]}")
        if query in EXPECTED:
            hits = len(EXPECTED[query].intersection(item_ids))
            recall[candidate_provider.model_id].append(hits / len(EXPECTED[query]))
for model_id, scores in recall.items():
    if scores:
        print(f"recall@5 {model_id}: {sum(scores) / len(scores):.2f} over {len(scores)} queries")

# To retrieve with Qwen3 in the cells below, uncomment:
# provider = local_provider


'something sweet and light'
  openai:text-embedding-3-small:768:retrieval-v1 (535 ms): ['Mango,Onion,Mint', 'Peshwari Naan', 'Mango Dip', 'Mint Dip', 'Plain Naan']
  qwen:Qwen/Qwen3-Embedding-0.6B:768:retrieval-v1 (1486 ms): ['Mango,Onion,Mint', 'Mango Dip', 'Garlic and Cheese Naan', 'Peshwari Naan', 'Chips']

'spicy chicken'
  openai:text-embedding-3-small:768:retrieval-v1 (497 ms): ['Chilli Chicken', 'Chicken Manchurian', 'Chicken Sag', 'Chicken Hot Garlic Resunga', 'Chicken Hariyali']
  qwen:Qwen/Qwen3-Embedding-0.6B:768:retrieval-v1 (99 ms): ['Chilli Chicken', 'Chicken Jalfrezi', 'Chicken Hariyali', 'Curry sauce', 'Karahi Chicken']

'a cold drink'
  openai:text-embedding-3-small:768:retrieval-v1 (1318 ms): ['Mint Dip', 'Mango,Onion,Mint', 'Lime pickle', 'Mango Dip', 'Chips']
  qwen:Qwen/Qwen3-Embedding-0.6B:768:retrieval-v1 (129 ms): ['Mango,Onion,Mint', 'Mint Dip', 'Mango Dip', 'Lime pickle', 'Curry sauce']

'vegetarian main course'
  openai:text-embedding-3-small:768:retrieval-v

## Retrieve with explicit constraints

Edit the semantic query and structured filters together. A price written only in `query` is not enforced. Unknown availability is excluded by default. Each candidate includes a cheapest valid required-choice configuration and its exact total; optional groups are omitted.

Vector and lexical scores help inspect ranking. They are not confidence values or a guarantee that an item meets a subjective preference.

In [13]:
query = "something spicy"
filters = RetrievalFilters(
    max_price=Decimal("20.00"),
    currency=menu.currency,
    category_ids=(),
    dietary_attributes=(),
    allow_unknown_availability=False,
)
candidates = retrieve_candidates(
    query, menu_id=menu.id, store=store, provider=provider,
    filters=filters, limit=5,
)
if not candidates:
    print("No candidates returned. No constraints were relaxed.")
for candidate in candidates:
    print(asdict(candidate))

{'item_id': '3ed6c3ad-9c06-45f6-a658-53a7f44bcb8f', 'variant_id': None, 'name': 'Chilli Chicken', 'variant_name': None, 'currency': 'GBP', 'minimum_price': Decimal('10.99'), 'required_choices': (), 'availability': 'available', 'score': 0.03200204813108039, 'semantic_score': 0.4280762843903939, 'lexical_score': 1.0}
{'item_id': '8809a5c0-31fd-4803-9af4-f3da493cf160', 'variant_id': None, 'name': 'Chicken Hariyali', 'variant_name': None, 'currency': 'GBP', 'minimum_price': Decimal('12.99'), 'required_choices': (), 'availability': 'available', 'score': 0.030303030303030304, 'semantic_score': 0.42264795550491363, 'lexical_score': 1.0}
{'item_id': '3b8e87d8-d788-4471-904d-02281e2a2d24', 'variant_id': None, 'name': 'Paneer chilli masala', 'variant_name': None, 'currency': 'GBP', 'minimum_price': Decimal('10.95'), 'required_choices': (), 'availability': 'available', 'score': 0.02889344262295082, 'semantic_score': 0.3816312308218448, 'lexical_score': 1.0}
{'item_id': '9cc178fc-bda1-485d-9d57-4a

## Inspect required choices

These selections explain the minimum price. They are not a placed order or a customer's final configuration.

In [11]:
items_by_id = {item.id: item for item in menu.items}
for candidate in candidates:
    item = items_by_id[candidate.item_id]
    groups = {group.id: group for group in item.variation_groups}
    print(candidate.name, candidate.variant_name or "", candidate.currency, candidate.minimum_price)
    for group_id, choice_ids in candidate.required_choices:
        group = groups[group_id]
        choices = {choice.id: choice for choice in group.choices}
        print(group.name, [(choices[cid].name, str(choices[cid].price_adjustment)) for cid in choice_ids])

Mango,Onion,Mint  GBP 1.8
Peshwari Naan  GBP 4.5
Mango Dip  GBP 0.8
Basmati Boiled Rice  GBP 2.95
Onion Dip  GBP 0.8


In [ ]:
# Close when finished; reconnect via the setup cell for further experiments.
store.connection.close()